In [ ]:
# Import path handling, pandas, and the project's stratified split helper.
from pathlib import Path

import pandas as pd

from support_sense.data_split import split_dataset

In [ ]:
# Point to the raw dataset using the path expected when running from notebooks/.
DATA_PATH = Path("../data/support_tickets.csv")

In [ ]:
# Read the source CSV and inspect that it can be loaded successfully.
df = pd.read_csv(DATA_PATH)

In [ ]:
# Load the complete labeled dataset.
df = pd.read_csv(DATA_PATH)

In [ ]:
# Produce the three reproducible project subsets.
train_df, validation_df, test_df = split_dataset(df)

In [ ]:
# Report the number of records allocated to model training.
print("Training:", len(train_df))

In [ ]:
# Report the number of records reserved for model selection.
print("Validation:", len(validation_df))

In [ ]:
# Report the number of records held out for final evaluation.
print("Test:", len(test_df))

In [ ]:
# Confirm that no records disappeared during splitting.
assert len(train_df) + len(validation_df) + len(test_df) == len(df)

In [ ]:
# Calculate category proportions in a stable alphabetical order.
def category_percentages(dataframe: pd.DataFrame) -> pd.Series:
    """Return category percentages for one dataset."""

    return dataframe["category"].value_counts(normalize=True).mul(100).sort_index()

In [ ]:
# Compare class proportions across the original and all three subsets.
print("Original")
print(category_percentages(df))

In [ ]:
# Display category percentages in the training subset.
print("\nTraining")
print(category_percentages(train_df))

In [ ]:
# Display category percentages in the validation subset.
print("\nValidation")
print(category_percentages(validation_df))

In [ ]:
# Display category percentages in the held-out test subset.
print("\nTest")
print(category_percentages(test_df))

In [ ]:
# Convert identifiers to sets so intersections are easy to test.
train_ids = set(train_df["ticket_id"])
validation_ids = set(validation_df["ticket_id"])
test_ids = set(test_df["ticket_id"])

In [ ]:
# No ticket may belong to both training and validation.
assert train_ids.isdisjoint(validation_ids)

In [ ]:
# No ticket may belong to both training and testing.
assert train_ids.isdisjoint(test_ids)

In [ ]:
# No ticket may belong to both validation and testing.
assert validation_ids.isdisjoint(test_ids)

In [ ]:
# Reconstruct the complete identifier set from all three subsets.
split_ids = train_ids | validation_ids | test_ids

In [ ]:
# Compare it with the raw dataset's identifier set.
original_ids = set(df["ticket_id"])

In [ ]:
# Verify that the three subsets contain every original ticket exactly once by ID.
assert split_ids == original_ids

## Split Summary

### Strategy

#### Explain why a 70/15/15 split was chosen.

A **70/15/15 split** assigns 210 of the 300 tickets to training, providing the model with most of the available examples while reserving separate 45-ticket validation and test sets. Validation supports model and hyperparameter selection; testing provides a final unbiased estimate after all choices are fixed. Because each evaluation subset contains only 7 or 8 examples per category, class-level metrics may be noisy and should be interpreted with uncertainty.

### Stratification

#### Explain why category was used as the stratification target.


`category` is the intended classification target and contains six equally represented classes. Stratifying on it prevents a random split from over- or under-representing a category and guarantees that every subset contains every category. Category-only stratification does not guarantee balanced `priority` or `sentiment` labels, so those distributions should be checked separately if they become prediction targets.

### Reproducibility

#### Explain why a fixed random state is used.

The fixed random state (`42`) produces the same ticket membership every time the split is recreated. This makes experiments, debugging, tests, and model comparisons repeatable; otherwise, score changes could come from different data assignments rather than from the model change being evaluated.

### Data Leakage

#### Explain what information the validation and test sets must not contribute during model training.

Validation and test text, labels, and derived statistics must not be used to fit the model or learned preprocessing steps. Vocabulary construction, vectorizer fitting, imputation, feature selection, resampling, class-weight estimation, and similar operations must be learned from the training set only. Validation data may guide model and hyperparameter selection, but the test set must not influence any choice.

The ticket-ID assertions pass, but they do not detect repeated content under different IDs. In the current deterministic split, `TKT-1184` is in training and `TKT-1159` is in testing even though their ticket text is identical. These records should be reviewed and grouped or deduplicated before the final split to avoid content leakage.

### Test Set Policy

#### Explain when the test set may be evaluated.

Evaluate the test set **once, after** preprocessing rules, features, model family, hyperparameters, decision thresholds, and reporting metrics have been finalized using only training and validation data. If test results are used to revise the system, that set has effectively become validation data and a new untouched test set is needed.

### Observed Distribution

#### Record the sizes and class distributions of all three datasets.

The split preserves all 300 ticket IDs with no ID overlap. The original data and training subset are exactly balanced; the odd-sized validation and test subsets differ by one record per category while complementing each other.

| Dataset | Rows | Share | `account_access` | `billing` | `cancellation_refund` | `delivery` | `product_inquiry` | `technical_issue` |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Original | 300 | 100% | 50 (16.67%) | 50 (16.67%) | 50 (16.67%) | 50 (16.67%) | 50 (16.67%) | 50 (16.67%) |
| Training | 210 | 70% | 35 (16.67%) | 35 (16.67%) | 35 (16.67%) | 35 (16.67%) | 35 (16.67%) | 35 (16.67%) |
| Validation | 45 | 15% | 8 (17.78%) | 8 (17.78%) | 7 (15.56%) | 7 (15.56%) | 7 (15.56%) | 8 (17.78%) |
| Test | 45 | 15% | 7 (15.56%) | 7 (15.56%) | 8 (17.78%) | 8 (17.78%) | 8 (17.78%) | 7 (15.56%) |